In [2]:
import csv

def cluster_by_sector(csv_filename):
    """
    Cluster job seekers by their sector attribute
    
    Args:
        csv_filename (str): Path to input CSV file
        
    Returns:
        dict: {sector_name: [list_of_seekers]}
    """
    sector_clusters = {}
    
    with open(csv_filename, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        
        for row in reader:
            sector = row['sector']
            
            if sector not in sector_clusters:
                sector_clusters[sector] = []
            
            sector_clusters[sector].append(row)
    
    return sector_clusters

# Example usage:
if __name__ == "__main__":
    clusters = cluster_by_sector("job_seekers_with_ids.csv")
    
    # Display cluster statistics
    print("Sector Clusters:")
    for sector, seekers in clusters.items():
        print(f"- {sector}: {len(seekers)} seekers")
        
    # Access full data for a specific sector
    print("\nSample Energy & Petroleum entries:")
    for seeker in clusters.get("Energy & Petroleum", [])[:1]:
        print(f"Name: {seeker['first_name']}")
        print(f"Skills: {seeker['technical_skills']}")
        print(f"Experience: {seeker['years_experience']} years\n")

Sector Clusters:
- Energy & Petroleum: 419 seekers
- E-Commerce: 420 seekers
- Media & Entertainment: 388 seekers
- Education: 409 seekers
- Construction: 392 seekers
- Aerospace & Defense: 399 seekers
- Renewable Energy: 386 seekers
- Telecommunications: 426 seekers
- Information Technology: 404 seekers
- Tourism & Hospitality: 405 seekers
- Manufacturing: 393 seekers
- Retail: 426 seekers
- Transport & Logistics: 409 seekers
- Banking & Finance: 393 seekers
- Agriculture: 409 seekers
- Healthcare: 410 seekers
- Pharmaceuticals: 429 seekers
- Fisheries: 383 seekers
- Mining: 415 seekers
- Public Sector: 409 seekers

Sample Energy & Petroleum entries:
Name: جمال الدّين
Skills: Microsoft Office, Critical Thinking, Team Leadership, Negotiation, Teamwork, Time Management, Report Writing, Arabic/French Bilingual, Adaptability
Experience: 9 years



In [4]:
import csv
import ast
from collections import defaultdict

class JobSeeker:
    def __init__(self, attributes):
        self.attributes = attributes
        self.sector = attributes['sector']
        # Handle missing/empty technical skills
        self.skills = set(attributes.get('technical_skills', '').split(', ') or ['general-skills'])
        # Convert years_experience to integer safely
        self.experience = int(attributes.get('years_experience', 0))
        # Handle edu_value conversion
        self.education = int(attributes.get('edu_value', 0))
        self.location = attributes.get('city', 'any')
        self.score = 0

    def __repr__(self):
        return f"{self.attributes['first_name']} ({self.sector})"

class JobRequirement:
    def __init__(self, name, value, priority, hard_constraint=True):
        self.name = name
        self.value = value
        self.priority = priority
        self.hard = hard_constraint

class SectorCSP:
    def __init__(self, job_reqs, clustered_seekers, k=3):
        # Sort requirements by priority first
        self.requirements = sorted(job_reqs, key=lambda x: x.priority)
        self.clusters = clustered_seekers
        self.k = k
        self.best_matches = []

    def safe_literal_eval(self, s):
        """Handle malformed JSON strings with single quotes"""
        try:
            return ast.literal_eval(s.replace("'", "\""))
        except:
            return {}

    def filter_cluster(self, current_cluster, requirement):
        filtered = []
        for seeker in current_cluster:
            match = False
            if requirement.name == 'sector':
                match = seeker.sector == requirement.value
            elif requirement.name == 'experience':
                match = seeker.experience >= requirement.value
            elif requirement.name == 'skills':
                match = requirement.value.issubset(seeker.skills)
            elif requirement.name == 'education':
                match = seeker.education >= requirement.value
            elif requirement.name == 'location':
                match = seeker.location == requirement.value
                
            if match:
                filtered.append(seeker)
        return filtered

    def evaluate_soft_constraints(self, seekers):
        for seeker in seekers:
            score = 0
            # Handle language proficiency parsing
            lang_str = seeker.attributes.get('language_proficiency', '{}')
            lang_proficiency = self.safe_literal_eval(lang_str)
            score += sum(1 for level in lang_proficiency.values() 
                        if level in ['Fluent', 'Native']) * 10
            
            # Education history parsing
            edu_str = seeker.attributes.get('education_history', '[]')
            education_history = self.safe_literal_eval(edu_str)
            score += len(education_history) * 5
            
            # Work history parsing
            work_str = seeker.attributes.get('work_history', '[]')
            work_history = self.safe_literal_eval(work_str)
            score += len(work_history) * 8
            
            seeker.score = score

    def backtrack(self, remaining_reqs, current_cluster, path=[]):
        if not remaining_reqs:
            self.evaluate_soft_constraints(current_cluster)
            current_sorted = sorted(current_cluster, key=lambda x: -x.score)[:self.k]
            if len(current_sorted) > len(self.best_matches):
                self.best_matches = current_sorted
            return

        current_req = remaining_reqs[0]
        filtered = self.filter_cluster(current_cluster, current_req)
        
        # Handle mandatory constraints
        if current_req.hard:
            if not filtered:
                return  # Prune invalid branch
            self.backtrack(remaining_reqs[1:], filtered, path + [current_req])
        else:
            # Explore both paths for soft constraints
            if filtered:
                self.backtrack(remaining_reqs[1:], filtered, path + [current_req])
            self.backtrack(remaining_reqs[1:], current_cluster, path)

    def solve(self):
        # Get initial sector cluster
        sector_req = next((req for req in self.requirements if req.name == 'sector'), None)
        if not sector_req:
            raise ValueError("Sector requirement is mandatory")
            
        initial_cluster = self.clusters.get(sector_req.value, [])
        self.backtrack(self.requirements, initial_cluster)
        return self.best_matches

def cluster_by_sector(csv_filename):
    """Read CSV and cluster seekers by sector"""
    clusters = defaultdict(list)
    
    with open(csv_filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                seeker = JobSeeker(row)
                clusters[seeker.sector].append(seeker)
            except Exception as e:
                print(f"Error processing row: {e}")
    
    return clusters

if __name__ == "__main__":
    # 1. Load and cluster data
    sector_clusters = cluster_by_sector("job_seekers_with_ids.csv")
    
    # 2. Define job requirements
    job_requirements = [
        JobRequirement('sector', 'Energy & Petroleum', 1, True),
        JobRequirement('experience', 5, 2, True),
        JobRequirement('skills', {'Critical Thinking', 'Team Leadership'}, 3, False),
        JobRequirement('education', 15, 4, False)
    ]

    # 3. Solve CSP
    csp = SectorCSP(job_requirements, sector_clusters, k=5)
    matches = csp.solve()

    # 4. Display results
    print("Top Candidates:")
    for i, seeker in enumerate(matches, 1):
        print(f"{i}. {seeker.attributes['first_name']}")
        print(f"   Sector: {seeker.sector}")
        print(f"   Experience: {seeker.experience} years")
        print(f"   Education Score: {seeker.education}")
        print(f"   Skills: {', '.join(seeker.skills)}")
        print(f"   Total Score: {seeker.score}\n")

Top Candidates:
1. طه
   Sector: Energy & Petroleum
   Experience: 29 years
   Education Score: 17
   Skills: Team Leadership, Teamwork, Project Management, Communication, Critical Thinking, Problem Solving, Strategic Planning, Conflict Resolution, Microsoft Office, Report Writing, Arabic/French Bilingual, Adaptability
   Total Score: 88

2. تاج
   Sector: Energy & Petroleum
   Experience: 29 years
   Education Score: 19
   Skills: Team Leadership, Teamwork, Project Management, Critical Thinking, Problem Solving, Strategic Planning, Conflict Resolution, Time Management, Microsoft Office, Report Writing, Arabic/French Bilingual, Adaptability
   Total Score: 88

3. ساطع
   Sector: Energy & Petroleum
   Experience: 28 years
   Education Score: 17
   Skills: Project Management, Reservoir Simulation, HYSYS, Communication, Critical Thinking, Problem Solving, Strategic Planning, Conflict Resolution, Microsoft Office, Well Logging, Team Leadership, Arabic/French Bilingual
   Total Score: 88

4